<a href="https://colab.research.google.com/github/sankeawthong/Project-1-Lita-Chatbot/blob/main/Lab02%20and%20Assignment_The_Honest_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 — The Honest Pipeline

**INT271 Machine Learning · Week 2**

| | |
|---|---|
| **Time budget** | ~2 hours (in-lab) |
| **Learning objectives** | 1. Clean real data (impute, justify) — 2. Build a leak-proof scikit-learn `Pipeline` (scale + encode + model) — 3. Estimate performance honestly with stratified splits and 5-fold CV — 4. Evaluate with confusion matrix, precision/recall/F1 and regression metrics — |
| **Dataset** | Titanic passenger records (891 rows) — real missing values, real categories, a real label |
| **Grading** | Exercises 1–6 + AI-usage disclosure cell · self-checks give instant feedback |

> **The story of this lab:** in the lecture you learned six words — *fit on train, transform both*. Today you build a pipeline that obeys them, measure honestly with cross-validation… and then, in the finale, you build a **deliberately leaky** workflow and watch it "beat" your honest one with numbers that are pure fiction. Once you catch a leak with your own hands, you never un-see it.

⚠️ **Before anything:** File → Save a copy in Drive, rename to `Lab2_<your_student_id>`.

---
## Setup — the standard course header

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (confusion_matrix, classification_report,
                             mean_absolute_error, mean_squared_error, r2_score)
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SelectKBest, f_classif

RNG_SEED = 42
np.random.seed(RNG_SEED)
print("numpy", np.__version__, "| pandas", pd.__version__)
print("Setup complete ✔")

> **A note on today's model:** we'll use `KNeighborsClassifier` (K-NN) as our classifier. You haven't been taught how it works — that's next week's lecture. Today it is a **black box with a `.fit()` and a `.predict()`**; everything in this lab is about what happens *around* the model, which is exactly the point.

---
## Part 1 · Load and audit

The Titanic dataset: 891 passengers, and the label `Survived` (1 = survived). Unlike the penguins, this data has *personality*: heavy missing values, text categories, and columns that shouldn't be features at all.

In [ ]:
TITANIC_URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(TITANIC_URL)
print(df.shape)
df.head()

In [ ]:
df.isna().sum()

**The audit, with lecture-grade justifications (this reasoning is examinable):**

| Column | Decision | Why |
|---|---|---|
| `Cabin` | **drop the column** | 687 of 891 missing (77%) — imputing would be mostly invention |
| `Age` | **keep, impute median** | 177 missing (20%) — too valuable to drop, median resists outliers |
| `Embarked` | **keep, impute most-frequent** | only 2 missing — any reasonable choice is fine |
| `PassengerId`, `Name`, `Ticket` | **drop** | identifiers, not features — a model that "learns" from ticket numbers is memorizing individuals (a leak wearing a costume!) |

Notice we **decide** the imputation strategy now but do **not** compute any imputation values yet. The means and medians will be *learned inside the pipeline, from training data only*. The bomb from the lecture — defused by design.

In [ ]:
df = df.drop(columns=["Cabin", "PassengerId", "Name", "Ticket"])

X = df.drop(columns=["Survived"])
y = df["Survived"]

numeric_features = ["Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Pclass", "Sex", "Embarked"]
print("Features:", list(X.columns))
print("Label balance:\n", y.value_counts(normalize=True).round(3))

38.4% survived — mildly imbalanced. Not Week-8 extreme, but enough to make `stratify` matter and to set up the accuracy paradox later.

*(Why is `Pclass` in the categorical list when it's a number? It's 1st/2nd/3rd class — a category wearing digits. One-hot treats it safely; treating it as a plain number would claim 3rd class is "three times" 1st class. Lecture Slide 12, in the wild.)*

---
## Part 2 · Split FIRST

Lecture rule: split **before** anything learns from the data. Shuffled, stratified, reproducible:

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RNG_SEED)

print("train:", X_train.shape, "| test:", X_test.shape)
print("survival rate — train:", y_train.mean().round(3), "| test:", y_test.mean().round(3))

Both halves have the same survival rate — that's `stratify=y` doing its job. **From this cell onward, `X_test` and `y_test` are locked in a vault.** They reappear exactly once, at the very end.

---
## Part 3 · The honest pipeline

Numeric and categorical columns need different treatment, so we build two mini-assembly-lines and join them with a `ColumnTransformer`, then bolt the model on the end. One object. One `.fit()`. Leak-proof by construction:

In [ ]:
numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale",  StandardScaler()),
])
categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])
model = Pipeline([
    ("prep", preprocess),
    ("knn",  KNeighborsClassifier(n_neighbors=7)),
])
model

Read the printed diagram bottom-up: when we call `model.fit(X_train, y_train)`, the imputers learn their medians/modes **from X_train**, the scaler learns its mean/std **from X_train**, the encoder learns its category list **from X_train** — and at prediction time, those learned values transform whatever comes in. *Fit on train, transform both* — enforced by the object itself.

---
## Part 4 · How lucky is a single split? Enter cross-validation

Before trusting any score, let's see the luck. Same model, same data — only the random split changes:

In [ ]:
for seed in [0, 1, 2, 3, 4]:
    Xtr, Xva, ytr, yva = train_test_split(X_train, y_train, test_size=0.25,
                                          stratify=y_train, random_state=seed)
    score = model.fit(Xtr, ytr).score(Xva, yva)
    print(f"random_state={seed}:  accuracy = {score:.3f}")

Same model — and the score moves by several points on pure luck of the draw. **Any single number above could have been the one you reported.** Cross-validation replaces the lottery with an average:

In [ ]:
cv_demo = cross_val_score(model, X_train, y_train, cv=5)   # stratified automatically for classifiers
print("fold scores:", cv_demo.round(3))
print(f"CV accuracy: {cv_demo.mean():.3f} ± {cv_demo.std():.3f}")

Mean **±** spread — the honest way to report performance while you're still developing. Note we cross-validate on the **training** data only; the vault stays shut.

---
## Part 5 · The mansions, revisited — regression metrics on a toy task

A quick detour to the other half of evaluation. Below: a synthetic housing market — a clean linear relationship, plus **three verified mansions** (remember Activity 1: we keep them). Watch what they do to the metrics:

In [ ]:
rng = np.random.default_rng(7)
size = rng.uniform(40, 220, 120)                      # m²
price = 30_000 * size + rng.normal(0, 300_000, 120)   # THB
size = np.append(size, [300, 320, 350])               # three mansions...
price = np.append(price, [45e6, 52e6, 60e6])          # ...at outlier prices

reg = LinearRegression().fit(size.reshape(-1, 1), price)
pred = reg.predict(size.reshape(-1, 1))

mae  = mean_absolute_error(price, pred)
rmse = np.sqrt(mean_squared_error(price, pred))
r2   = r2_score(price, pred)
print(f"MAE  = {mae/1e6:.2f} M THB   (average miss)")
print(f"RMSE = {rmse/1e6:.2f} M THB   (big misses amplified)")
print(f"R²   = {r2:.3f}")

plt.figure(figsize=(7, 4))
plt.scatter(size, price/1e6, alpha=0.7)
plt.plot(np.sort(size), reg.predict(np.sort(size).reshape(-1,1))/1e6, color="crimson")
plt.xlabel("Size (m²)"); plt.ylabel("Price (M THB)"); plt.title("A market with three mansions")
plt.show()

RMSE towers over MAE — the lecture's tell that **a few huge errors are hiding in the model**. Three points out of 123 dominate one metric and barely touch the other. Which metric is "right"? Wrong question. They answer different questions; *choosing* is your job.

---
## Part 6 · The finale (guided): watching a leak lie to you

You don't have to write this code — **run it and read it carefully.** Here is the setup: `X_noise` is **pure random noise** — 2,000 meaningless features — and `y_coin` is a **coin flip**. There is *nothing to learn*. Any honest method must score ≈ 0.50.

Now we do something that looks innocent: *"we have too many features — let's keep only the 20 most correlated with the label"* … computed on the **full dataset**, before cross-validating. Sound familiar? It's Activity 2, case A — the imputation leak's big brother.

In [ ]:
n, p = 200, 2000
X_noise = pd.DataFrame(np.random.default_rng(0).normal(size=(n, p)))
y_coin  = pd.Series(np.random.default_rng(1).integers(0, 2, n))

# --- LEAKY: select features using ALL the data, then cross-validate ---
selector = SelectKBest(f_classif, k=20).fit(X_noise, y_coin)      # <-- the crime
X_selected = selector.transform(X_noise)
leaky_scores = cross_val_score(KNeighborsClassifier(7), X_selected, y_coin, cv=5)

# --- HONEST: selection lives INSIDE the pipeline, so it's re-fit per fold ---
honest_model = Pipeline([("select", SelectKBest(f_classif, k=20)),
                         ("knn", KNeighborsClassifier(7))])
honest_scores = cross_val_score(honest_model, X_noise, y_coin, cv=5)

print(f"LEAKY  workflow on pure noise: {leaky_scores.mean():.3f} ± {leaky_scores.std():.3f}")
print(f"HONEST workflow on pure noise: {honest_scores.mean():.3f} ± {honest_scores.std():.3f}")
print("\nReminder: the true predictability of a coin flip is 0.500")

**Read those numbers again.** The leaky workflow reports impressive "accuracy" at predicting a *coin flip* — a thing that cannot be predicted. Every point above 0.50 is fiction, manufactured by letting the feature selector peek at the whole dataset. The honest pipeline — identical selector, identical model, just fitted *inside* each fold — tells the truth: there's nothing here.

This exact mistake has appeared in published research. You are now immune. The vaccine cost you one code cell.

---
---
# Exercises (1st Assignment)

Same rules as Lab 1: replace `...`, run each ✅ self-check, disclose AI use at the end, be ready to explain any line orally.

### Exercise 1 — Audit arithmetic
From the **original** Titanic file (reload it below so column drops don't interfere), compute:
- **`pct_cabin_missing`** — the percentage of rows missing `Cabin`, rounded to 1 decimal (e.g., `43.2`)
- **`n_age_missing`** — the number of rows missing `Age` (an integer)

In [ ]:
df_raw = pd.read_csv(TITANIC_URL)
pct_cabin_missing = ...
n_age_missing = ...
print(pct_cabin_missing, n_age_missing)

In [ ]:
# ✅ Self-check — Exercise 1
assert n_age_missing == 177, "Count the missing Age values from df_raw."
assert abs(pct_cabin_missing - 77.1) < 0.2, "Cabin missing %: (missing / total) × 100, rounded to 1 dp."
print("Exercise 1 self-check passed ✔")

### Exercise 2 — A split you can defend
Make a fresh split of `X`, `y` (the cleaned versions from Part 1) named **`Xtr2, Xte2, ytr2, yte2`** with: 20% test size, stratified on the label, `random_state=0`. Then compute **`rate_gap`** — the absolute difference between the survival rates of `ytr2` and `yte2`.

In [ ]:
Xtr2, Xte2, ytr2, yte2 = ...
rate_gap = ...
print(Xtr2.shape, Xte2.shape, '| gap:', round(rate_gap, 4))

In [ ]:
# ✅ Self-check — Exercise 2
assert Xte2.shape[0] == 179, "20% of 891 rows should give a 179-row test set."
assert rate_gap < 0.01, "With stratify=y the survival-rate gap should be tiny (<0.01)."
print("Exercise 2 self-check passed ✔")

### Exercise 3 — Your own honest pipeline
Build **`my_model`**: a `Pipeline` exactly like Part 3's, but with **mean** imputation for numeric features (instead of median) and **`n_neighbors=5`**. Then cross-validate it (5-fold) on `X_train, y_train`, storing the scores in **`my_cv`**.

*Reuse the `categorical_pipe` idea from Part 3 — but build fresh objects; don't recycle the fitted ones.*

In [ ]:
num2 = ...
cat2 = ...
prep2 = ...
my_model = ...
my_cv = ...
print(my_cv.round(3), '| mean:', my_cv.mean().round(3))

In [ ]:
# ✅ Self-check — Exercise 3
assert isinstance(my_model, Pipeline), "my_model must be a sklearn Pipeline."
assert my_model.get_params()["knn__n_neighbors"] == 5, "Set n_neighbors=5 on the KNN step (name it 'knn')."
assert my_model.get_params()["prep__num__impute__strategy"] == "mean", "Numeric imputer should use strategy='mean' (steps named prep/num/impute)."
assert len(my_cv) == 5 and 0.70 < my_cv.mean() < 0.90, "5 folds expected, mean accuracy in a sane range."
print("Exercise 3 self-check passed ✔")

### Exercise 4 — Opening the vault: final evaluation
Development is over — time to spend the test set, **once**. Using the Part 3 `model` (n_neighbors=7, median imputation):
1. Fit it on the **full training set** (`X_train, y_train`)
2. Predict the test set into **`y_pred`**
3. Build **`cm`** — the confusion matrix (`confusion_matrix(y_test, y_pred)`)
4. From `classification_report(y_test, y_pred, output_dict=True)`, extract the **recall of the survivor class** (key `"1"`) into **`recall_survived`**
5. Fit a `DummyClassifier(strategy="most_frequent")` on the training data and store its test accuracy in **`dummy_acc`** — the "predict nobody survived" baseline from the lecture

In [ ]:
model.fit(...)
y_pred = ...
cm = ...
report = classification_report(y_test, y_pred, output_dict=True)
recall_survived = ...
dummy_acc = ...
print(cm)
print('recall (survived):', round(recall_survived, 3), '| dummy accuracy:', round(dummy_acc, 3))
print('model accuracy:', round(model.score(X_test, y_test), 3))

In [ ]:
# ✅ Self-check — Exercise 4
assert cm.shape == (2, 2) and cm.sum() == len(y_test), "Confusion matrix should cover every test passenger."
assert 0.5 < recall_survived < 0.95, "Extract recall for class '1' from the report dict."
assert 0.55 < dummy_acc < 0.68, "The most-frequent dummy should score ≈ the majority-class rate."
print("Exercise 4 self-check passed ✔")
print(f"\nThe accuracy paradox, live: the do-nothing dummy scores {dummy_acc:.0%} — respectable-looking, survivors found: 0.")

### Exercise 5 — Reading the confusion matrix like an engineer
Using **your `cm` from Exercise 4** (layout: row 0 = actually died, row 1 = actually survived; column 0 = predicted died, column 1 = predicted survived), extract as integers:
- **`fn`** — survivors the model missed (predicted died, actually survived)
- **`fp`** — false alarms (predicted survived, actually died)

Then set **`worse_for_rescue`** to `"fn"` or `"fp"`: *if this model triaged who gets a lifeboat-search first, which error costs lives?*

In [ ]:
fn = ...
fp = ...
worse_for_rescue = '...'
print('missed survivors (FN):', fn, '| false alarms (FP):', fp)

In [ ]:
# ✅ Self-check — Exercise 5
assert fn == cm[1, 0] and fp == cm[0, 1], "Check the row/column meanings: rows = actual, columns = predicted."
assert worse_for_rescue == "fn", "Think it through: a missed survivor is never searched for."
print("Exercise 5 self-check passed ✔")

### Exercise 6 — Improve it honestly (open-ended)
Try to beat the Part 3 pipeline's CV score — **honestly**. Pick ONE change and implement it in a fresh pipeline `improved_model`:
- engineer a feature (classic: `FamilySize = SibSp + Parch + 1` — add it to `X` and to `numeric_features` for your pipeline), **or**
- change the imputation strategy, **or**
- change `n_neighbors`.

Then: (1) report its 5-fold CV mean ± std on the training data next to the original's, and (2) in the Markdown cell below, write **3+ sentences**: what you changed, what happened, and *why the comparison is trustworthy* (which rules from today's lecture protect it?).

In [ ]:
improved_model = ...
# ... your comparison here: CV mean ± std for both models, on training data only
...

*Your write-up (≥ 3 sentences): what you changed · what happened · why the comparison is trustworthy.*

✍️ ...

---
## AI-usage disclosure (required)

> **AI tools used:** *e.g., "ChatGPT — explained ColumnTransformer syntax; helped debug a KeyError in Exercise 3" — or — "None."*
>
> **Written and understood by:** *your name & student ID*
>
> I can explain every line of code in this notebook.

## 📤 Submitting

1. **Runtime → Restart session and run all** — every cell runs clean, every self-check prints *passed*.
2. Disclosure filled, notebook renamed `Lab2_<student_id>`.
3. **File → Download → .ipynb** → upload to the LMS before the deadline.


---
*Dataset: Titanic passenger manifest (public domain), mirrored by Data Science Dojo.*